In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import llm, prompts
from importlib import reload

In [ ]:
reload(llm)

In [ ]:
#DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables'
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables'

In [ ]:
orig_df = pd.read_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\final_paul_imp_ntf_no_dups_22_1_with_eligibility.xlsx')
orig_df.head()

In [ ]:
merged = orig_df

In [ ]:
missing = is_missing(merged.year) | is_missing(merged.abstract)
has_pmid = ~is_missing(merged.PMID)
(has_pmid & missing).sum()

In [ ]:
titles = list(merged.loc[is_missing(merged.PMID), 'title'])
len(titles)

In [ ]:
eligibility_df = llm.check_eligibility(
    orig_df,
    model=llm.GEMINI_3_FLASH,
    system_prompt=prompts.SYSTEM_PROMPT,
    criteria=prompts.PAUL_CRITERIA,
    suffix='_title_abstract_gemini',
    use_abstract=True,
)
eligibility_df.decision_title_abstract_gemini.sum()

In [ ]:
eligibility_df = llm.check_eligibility(
    eligibility_df,
    model=llm.GEMINI_3_FLASH,
    system_prompt=prompts.SYSTEM_PROMPT,
    criteria=prompts.PAUL_CRITERIA,
    suffix='_title_only_gemini',
    use_abstract=False,
)
eligibility_df.decision_title_only_gemini.sum()

In [ ]:
eligibility_df = llm.check_eligibility(
    eligibility_df,
    model=llm.GPT_MINI,
    criteria=llm.JUMALAH_CRITERIA,
    suffix='_title_abstract_gpt',
    use_abstract=True,
)
eligibility_df.decision_title_abstract_gpt.sum()

In [ ]:
eligibility_df = llm.check_eligibility(
    eligibility_df,
    model=llm.GPT_MINI,
    criteria=llm.JUMALAH_CRITERIA,
    suffix='_title_only_gpt',
    use_abstract=False,
)
eligibility_df.decision_title_only_gpt.sum()

In [ ]:
eligibility_df.head()

In [ ]:
eligibility_df.to_excel(DIR + r'\compare_4.xlsx', index=False)

In [ ]:
sub = [col for col in eligibility_df.columns if col.startswith('decision_')]
eligibility_df[sub].sum()

In [ ]:
cols = list(orig_df.columns)
cols.remove('abstract')
cols

In [ ]:
partial_df = orig_df[cols]
title_eligibility_df = 

In [ ]:
eligibility_df.to_excel(os.path.join(DIR, 'deduplicate_with_eligibility.xlsx'), index=False)

In [ ]:
FUCKING_CRITERIA = '''
## Inclusion criteria

1.	Original research articles of peer-reviewed observational studies; cohort studies, cross-sectional studies, and case-control studies. Other study designs including case reports, case series, RCTs and reviews of all kinds should be excluded. 
2.	Studies that report on participants who tested for SARS-CoV-2 through serological tests (rapid tests) or molecular tests (PCR) irrespective of inclusion or exclusion of control groups. – Include even if the study did not have a control group. 
3.	Studies should be included if they report self reported or verified symptoms of SARS-COV-2 infection after symptoms of the initial SARS-COV-2 infection have subsided , or new symptoms not SARS-COV-2 infection related or ongoing SARS-COV-2 infection symptoms after the acute phase. (Ensure the study report symptoms of long Covid). 
4.	No time restriction will be applied on the time of onset of the symptoms provided they are symptoms appearing on recovered patients . (Be keen on timing of onset, not to include studies describing acute phase symptoms- must be long covid symptoms).
5.	Studies reporting on the mental health consequences of SARS-COV-2 infection should also be included . (Anxiety, depression, mood disorders and symptoms, cognitive dysfunction symptoms, memory loss, loss of concentration etc.)
'''

In [ ]:
eligibility_df = eligibility_df.copy()

In [ ]:
eligibility_df.loc[eligibility_df.decision, 'decision'] = np.nan

In [ ]:
gpt.fill_in_eligibility(eligibility_df, criteria=FUCKING_CRITERIA)

In [ ]:
eligibility_df.to_excel(os.path.join(DIR, 'deduplicate_with_eligibility_2.xlsx'), index=False)